In [ ]:
# Example 1: version + basic package attributes.
# jupyter_offlinenotebook does not export __version__ at module level —
# read the installed distribution version via importlib.metadata instead.
from importlib.metadata import version

import jupyter_offlinenotebook

dist_version = version("jupyter-offlinenotebook")
print(f"jupyter_offlinenotebook version: {dist_version}")

# Verify version is a non-empty string in expected format
assert isinstance(dist_version, str)
assert len(dist_version) > 0
assert dist_version[0].isdigit(), f"Version should start with a digit, got: {dist_version}"

# The core server-extension classes are importable at top level
assert hasattr(jupyter_offlinenotebook, "OfflineNotebookConfig")
assert hasattr(jupyter_offlinenotebook, "OfflineNotebookHandler")
print("OfflineNotebookConfig + OfflineNotebookHandler present")


In [ ]:
# Example 2: package layout inspection (jupyter_offlinenotebook is a Jupyter server extension)
import os
import jupyter_offlinenotebook

pkg_dir = os.path.dirname(jupyter_offlinenotebook.__file__)
print(f"Package directory: {pkg_dir}")

contents = sorted(os.listdir(pkg_dir))
print(f"Contents ({len(contents)} entries):")
for entry in contents:
    print(f"  {entry}")

assert "__init__.py" in contents, "Expected __init__.py in package directory"

In [ ]:
# Example 3: Jupyter server extension hook discovery
# Server extensions expose _jupyter_server_extension_points() (or _paths in older API).
import jupyter_offlinenotebook

hook_names = [
    "_jupyter_server_extension_points",
    "_jupyter_server_extension_paths",
    "_jupyter_nbextension_paths",
    "load_jupyter_server_extension",
]

found = {name: hasattr(jupyter_offlinenotebook, name) for name in hook_names}
for name, present in found.items():
    print(f"  {name}: {'present' if present else 'absent'}")

# At least one of the extension hooks should be defined
assert any(found.values()), "Expected at least one Jupyter extension hook"

In [ ]:
# Example 4: invoke the extension-points hook to inspect declared modules
import jupyter_offlinenotebook

result = None
if hasattr(jupyter_offlinenotebook, "_jupyter_server_extension_points"):
    result = jupyter_offlinenotebook._jupyter_server_extension_points()
elif hasattr(jupyter_offlinenotebook, "_jupyter_server_extension_paths"):
    result = jupyter_offlinenotebook._jupyter_server_extension_paths()

print(f"Server extension points: {result}")
assert isinstance(result, list) and len(result) > 0
for entry in result:
    assert isinstance(entry, dict)
    assert "module" in entry

In [ ]:
# Example 5: nbextension paths (frontend JS bundle declaration)
import jupyter_offlinenotebook

nbext = None
if hasattr(jupyter_offlinenotebook, "_jupyter_nbextension_paths"):
    nbext = jupyter_offlinenotebook._jupyter_nbextension_paths()
    print(f"nbextension paths: {nbext}")
    assert isinstance(nbext, list) and len(nbext) > 0
    for entry in nbext:
        assert isinstance(entry, dict)
        # Required keys per Jupyter spec
        for key in ("section", "src", "dest", "require"):
            assert key in entry, f"Missing key {key} in nbextension entry"
    print("nbextension paths schema OK")
else:
    print("No _jupyter_nbextension_paths hook (acceptable for newer Jupyter server-only extensions)")

print("\njupyter_offlinenotebook smoke tests passed")